# Agente de Retinografia — servidor de inferência (Colab)

Este notebook liga o **servidor** que o aplicativo do celular usa. Ele carrega **os mesmos modelos do notebook EyeFlow V800** (RETFound + ensemble de 3 + MedGemma-27B com LoRA), lidos do seu Drive em `EyeFlow/glaucoma_cot_v800`, e cria um endereço público **https** para o celular enviar as retinografias.

**Antes de executar**
1. *Ambiente de execução → Alterar o tipo* → **GPU A100** (para só o Stage 1, qualquer GPU serve: defina `USAR_STAGE2 = False`).
2. Segredo `HF_TOKEN` configurado (ícone da chave 🔑 à esquerda), o mesmo usado no notebook V800.
3. Execute as células em ordem (*Ambiente de execução → Executar tudo*).

No fim aparecem o **endereço do servidor**, a **chave de acesso** e um **QR code** que já configura o app.

> Deixe esta aba aberta enquanto usa o aplicativo. Quando o Colab desconecta, o servidor para, e o endereço muda na próxima vez.

In [ ]:
# ==========================================================================================
# CÉLULA 1 — OPÇÕES
# ==========================================================================================
USAR_STAGE2 = True        # False = só Stage 1 (laudo em segundos, qualquer GPU)
REPO_GIT    = ""          # ex.: "https://github.com/SEU-USUARIO/agente-retinografia.git" (vazio = usar a cópia no Drive)
PASTA_DRIVE = "/content/drive/MyDrive/EyeFlow/agente-retinografia"   # onde está a pasta do projeto no Drive
CHAVE_FIXA  = ""          # deixe vazio para gerar uma chave nova a cada sessão

from google.colab import drive
drive.mount("/content/drive")
import os, secrets, subprocess, sys, time, json, re
from pathlib import Path
CHAVE = CHAVE_FIXA or secrets.token_urlsafe(9)
if REPO_GIT:
    subprocess.run(["rm","-rf","/content/agente-retinografia"])
    subprocess.run(["git","clone","--depth","1",REPO_GIT,"/content/agente-retinografia"],check=True)
    PROJ = Path("/content/agente-retinografia")
else:
    PROJ = Path(PASTA_DRIVE)
assert (PROJ/"servidor"/"app.py").exists(), f"Não encontrei {PROJ}/servidor/app.py — confira PASTA_DRIVE ou REPO_GIT."
print("Projeto:", PROJ)

In [ ]:
# ==========================================================================================
# CÉLULA 2 — DEPENDÊNCIAS (mesmas versões do V800) + túnel Cloudflare
# ==========================================================================================
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJ/"servidor"/"requirements.txt")],check=True)
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"],check=False)
if not Path("/usr/local/bin/cloudflared").exists():
    subprocess.run(["wget","-q","-O","/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],check=True)
    subprocess.run(["chmod","+x","/usr/local/bin/cloudflared"],check=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

In [ ]:
# ==========================================================================================
# CÉLULA 3 — LIGA O SERVIDOR E CARREGA OS MODELOS (≈ 3–4 min com o MedGemma-27B)
# ==========================================================================================
from google.colab import userdata
env = dict(os.environ, EYEFLOW_ROOT="/content/drive/MyDrive/EyeFlow", USAR_STAGE2="1" if USAR_STAGE2 else "0",
           AGENTE_CHAVE=CHAVE, HF_TOKEN=(userdata.get("HF_TOKEN") if USAR_STAGE2 else "") or "",
           HF_DEACTIVATE_ASYNC_LOAD="1", HF_ENABLE_PARALLEL_LOADING="false",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")
subprocess.run("pkill -f 'uvicorn app:app'", shell=True)
log = open("/content/servidor.log","w")
srv = subprocess.Popen([sys.executable,"-m","uvicorn","app:app","--host","0.0.0.0","--port","8000"],
                       cwd=str(PROJ/"servidor"), env=env, stdout=log, stderr=subprocess.STDOUT)
import urllib.request
t0=time.time()
while True:
    time.sleep(10)
    try:
        req=urllib.request.Request("http://127.0.0.1:8000/api/saude",headers={"X-Chave":CHAVE})
        s=json.loads(urllib.request.urlopen(req,timeout=5).read())
    except Exception as e:
        s={"pronto":False,"mensagem":"iniciando…"}
    print(f"[{time.time()-t0:5.0f} s] {s.get('mensagem') or ('pronto' if s.get('pronto') else '')} {s.get('erro','')}")
    if s.get("pronto") or s.get("erro") or srv.poll() is not None: break
if not s.get("pronto"):
    print(open("/content/servidor.log").read()[-4000:]); raise RuntimeError("Servidor não ficou pronto (veja o log acima).")
print("[OK] Modelos carregados:", {k:s[k] for k in ("stage2","dispositivo","carregado_em_s")})

In [ ]:
# ==========================================================================================
# CÉLULA 4 — ENDEREÇO PÚBLICO HTTPS (Cloudflare Tunnel) + QR CODE DE CONFIGURAÇÃO
# ==========================================================================================
APP_WEB = ""   # opcional: endereço do app no GitHub Pages, ex.: "https://SEU-USUARIO.github.io/agente-retinografia/"
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
tlog=open("/content/tunel.log","w")
tun=subprocess.Popen(["cloudflared","tunnel","--no-autoupdate","--url","http://127.0.0.1:8000"],stdout=tlog,stderr=subprocess.STDOUT)
URL=None
for _ in range(60):
    time.sleep(1)
    m=re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",open("/content/tunel.log").read())
    if m: URL=m.group(0); break
if not URL: raise RuntimeError("Túnel não abriu:\n"+open("/content/tunel.log").read()[-2000:])
from urllib.parse import quote
link=(APP_WEB or URL+"/")+f"?servidor={quote(URL,safe='')}&chave={quote(CHAVE,safe='')}"
import qrcode
from IPython.display import display, HTML
qr=qrcode.make(link); qr.save("/content/qr_agente.png")
print("ENDEREÇO DO SERVIDOR :", URL)
print("CHAVE DE ACESSO      :", CHAVE)
print("LINK DE CONFIGURAÇÃO :", link)
display(HTML("<p><b>Aponte a câmera do celular para o QR code</b>: ele abre o app já configurado.<br>"
             "No app Android (APK), vá em <b>Ajustes</b> e digite o endereço e a chave acima.</p>"))
display(qr.get_image() if hasattr(qr,"get_image") else qr)

In [ ]:
# ==========================================================================================
# CÉLULA 5 — MANTER LIGADO E ACOMPANHAR (pare esta célula para encerrar)
# ==========================================================================================
from IPython.display import clear_output
try:
    while True:
        clear_output(wait=True)
        print("Servidor:",URL,"| chave:",CHAVE,"| ativo" if srv.poll() is None else "| PAROU")
        print("".join(open("/content/servidor.log").readlines()[-15:]))
        time.sleep(30)
except KeyboardInterrupt:
    tun.terminate(); srv.terminate(); print("Servidor encerrado.")

### Observações
- **Privacidade:** as imagens ficam só na memória do servidor e o resultado é apagado após 1 hora. O histórico de laudos fica apenas no celular.
- **Tempo por exame:** cerca de 1–2 min na análise completa (Stage 1 + Stage 2 + tradução) e poucos segundos na análise rápida (só Stage 1), iguais aos do notebook V800.
- **Fila:** se dois celulares enviarem ao mesmo tempo, a GPU processa um exame por vez.
- **Uso contínuo:** o Colab desliga por inatividade e o endereço `trycloudflare.com` muda a cada sessão. Para um servidor fixo 24 h, veja a seção *Servidor permanente* do GUIA.